# Demo 06A · Notebook 2: Facilitator AgentCore Walkthrough

**Facilitator-only.** This notebook invokes and inspects a **pre-deployed** AgentCore Runtime, Gateway, and reservation Lambda running against the canonical event graph. It creates and deploys nothing. Participants observe this walkthrough; they do not run it.

The same `HybridCypherRetriever` contract from Notebook 1 runs inside the Runtime. The agent has exactly two logical tools: the in-Runtime `search_hotel_knowledge` retrieval tool and one `create_reservation_request` command exposed through Gateway. There is no payment, confirmation, cancellation, or inventory in this design.

In [ ]:
import json
import os
import uuid

import boto3
from dotenv import load_dotenv

from workshop.contracts import MAX_GUESTS, OVER_LIMIT_GUESTS
from workshop.graph_setup import HERO_NAME

load_dotenv()

RUNTIME_ARN = os.getenv("AGENT_RUNTIME_ARN", "").strip()
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
BEDROCK_READY = boto3.Session().get_credentials() is not None
RUNTIME_READY = bool(RUNTIME_ARN) and BEDROCK_READY

NEO4J_ENV = ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD", "NEO4J_DATABASE")
NEO4J_READY = all(os.getenv(name) for name in NEO4J_ENV)


def invoke_runtime(prompt, request_id):
    client = boto3.client("bedrock-agentcore", region_name=AWS_REGION)
    response = client.invoke_agent_runtime(
        agentRuntimeArn=RUNTIME_ARN,
        payload=json.dumps(
            {"prompt": prompt, "request_id": request_id}
        ).encode("utf-8"),
    )
    return json.loads(response["response"].read())


if not RUNTIME_READY:
    print("No pre-deployed Runtime configured (set AGENT_RUNTIME_ARN with AWS credentials).")
    print("All live cells below will be skipped. This is expected in repository validation.")
else:
    print(f"Facilitator Runtime configured in {AWS_REGION}.")

## 1. One caller-created request ID, reused on retries

The caller creates a single canonical UUID and reuses it for every delivery of the same reservation request. It is the idempotency key and the correlation identifier across Runtime, Gateway, the reservation Lambda, and Neo4j-related work.

In [ ]:
REQUEST_ID = str(uuid.uuid4())
print(f"Caller-created request_id (reuse on retries): {REQUEST_ID}")

## 2. A 15-guest request is rejected with no write

The Neo4j maximum-guests rule caps a reservation request at 10 guests. The command enforces the rule inside the same boundary as the write, so an over-limit request is rejected and nothing is written to the graph.

In [ ]:
if not RUNTIME_READY:
    print("Skipping 15-guest rejection: no Runtime configured.")
else:
    prompt = (
        f"Find {HERO_NAME} and create a reservation request for "
        f"{OVER_LIMIT_GUESTS} guests, check-in 2026-09-04, check-out 2026-09-06."
    )
    result = invoke_runtime(prompt, REQUEST_ID)
    print(json.dumps(result, indent=2))

## 3. A corrected request within the limit is recorded

Reusing the same `request_id`, the facilitator submits a request within the 10-guest limit. The command creates one `ReservationRequest` linked to the retrieved hotel by a `FOR_HOTEL` relationship. Re-delivering the identical request returns the existing record without creating a duplicate.

In [ ]:
if not RUNTIME_READY:
    print("Skipping corrected request: no Runtime configured.")
else:
    prompt = (
        f"Find {HERO_NAME} and create a reservation request for "
        f"{MAX_GUESTS} guests, check-in 2026-09-04, check-out 2026-09-06."
    )
    result = invoke_runtime(prompt, REQUEST_ID)
    print(json.dumps(result, indent=2))

## 4. Inspect the resulting request in the graph

Anchored on the stable `request_id`, confirm exactly one accepted request linked to exactly one hotel.

In [ ]:
if not NEO4J_READY:
    print("Skipping graph inspection: Neo4j is not configured.")
else:
    from neo4j import GraphDatabase

    driver = GraphDatabase.driver(
        os.environ["NEO4J_URI"],
        auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]),
    )
    query = (
        "MATCH (r:ReservationRequest {request_id: $rid})-[:FOR_HOTEL]->(h:Hotel) "
        "RETURN r.status AS status, r.guests AS guests, r.check_in AS check_in, "
        "r.check_out AS check_out, h.hotel_id AS hotel_id, h.name AS hotel_name, "
        "toString(r.created_at) AS created_at"
    )
    try:
        with driver.session(database=os.environ["NEO4J_DATABASE"]) as session:
            for record in session.run(query, rid=REQUEST_ID):
                print(dict(record))
    finally:
        driver.close()

## 5. Correlate AgentCore and CloudWatch by request ID

Every Runtime and Lambda log line records the `request_id` (never prompts, credentials, or connection strings). Use it to trace one reservation across Runtime retrieval, Gateway, the reservation Lambda, and Neo4j-related work. AgentCore Runtime logs land in Amazon CloudWatch under `/aws/bedrock-agentcore/runtimes/`.

In [ ]:
if not RUNTIME_READY:
    print("Skipping log correlation: no Runtime configured.")
else:
    logs = boto3.client("logs", region_name=AWS_REGION)
    print(f"Search CloudWatch Logs Insights for request_id={REQUEST_ID}")
    print("Log group prefix: /aws/bedrock-agentcore/runtimes/")
    print("Example filter pattern:")
    print(f'  fields @timestamp, @message | filter @message like "{REQUEST_ID}" | sort @timestamp asc')
    # A live facilitator can uncomment a filter_log_events call here against the
    # specific runtime and Lambda log groups for their deployment.

## Facilitator notes

**Expected outcomes**
- 15-guest request: `status="rejected"`, `reason_code="max_guests_exceeded"`, and no new graph node.
- Corrected request: `status="accepted"` with a `created_at` timestamp and exactly one `FOR_HOTEL` relationship.
- Re-running the corrected cell with the same `REQUEST_ID`: `duplicate=true`, same timestamp, no second node.

**Recovery**
- If a live cell errors, confirm `AGENT_RUNTIME_ARN`, region, and Bedrock model access, then re-run. Cells are safe to re-run; the command is idempotent by `request_id`.
- To start a clean scenario, restart the kernel so a fresh `REQUEST_ID` is generated.

**Where this goes next**
- **Demo 09** teaches Neo4j MCP and controlled Text2Cypher as a separate advanced path.
- **Demo 06B** (deferred, optional) builds the from-scratch AWS deployment that produced the Runtime, Gateway, and Lambda used here.